In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import (
    CarSim,
)
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)
sim = CarSim(prop)

In [ ]:
class Tutorial2(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.4, 1.5), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 2.0)
        self.random_d_xy = (0.1, 0.2)
        self.random_d_yaw_deg = 1
        self.set_signs(
            [
                Sign(x=1.0, y=2.0, name="sign1"),
                Sign(x=1.8, y=1.8, name="sign1"),
                Sign(x=2.6, y=1.4, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, rotate, search, **kwargs):
        """回答例：回転と直進を交互にして認識対象に近づく"""
        while True:
            pos = search()
            if pos is None:
                move(v=0)
            else:
                ######## ここから下にプログラムを書こう
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=1)
                ######## ここより上にプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう


sim.set_mission(Tutorial2())
sim.run()
SimDrawer(sim).show()